## COPIA of TareaM1.ipynb

En esta tarea vamos a estudiar las estadísticas de un robot de limpieza reactivo.

Dado:

Habitación de MxN espacios.

Número de agentes.

Porcentaje de celdas inicialmente sucias.

Tiempo máximo de ejecución.

Podremos analizar que tan eficientes pueden ser las aspiradoras robóticas en un tiempo estimado

de limpieza y de esta manera con diferentes casos de prueba llegar a una conclusión

sobre la importancia de los multiagentes activos en nuestro entorno.

In [1]:
import agentpy as ap
import numpy as np
from matplotlib import pyplot as plt
from IPython.display import HTML

#PARA LOS COMENTARIOS EN EL CÓDIGO SE UTILIZÓ LA AYUDA DE LA IA
#POR EL MOTIVO DE DAR UN MEJOR ENTENDIMIENTO AL CÓDIGO CON PALABRAS TÉCNICAS
#Cabe recalcar, para tener una mejor visualización de las simulaciones
#también me ayudé de la IA para que me ayude a tener una mejor representación para las cuadrículas y agentes.


# Agente Aspiradora

# Agente Aspiradora

class VacuumAgent(ap.Agent):

    def setup(self):
        # Cada agente se inicializa conociendo su entorno (enviroment = habitación)
        self.enviroment = self.model.enviroment
        # Bandera que indica si limpió en el turno actual
        self.cleaned = False

    def getpos(self):
        # Retorna la posición actual del agente en la cuadrícula como un array numpy
        return np.array(self.enviroment.positions[self])

    def moverandom(self):
        # Obtiene la posición actual del agente
        pos = tuple(self.getpos())
        # Lista de posibles movimientos (8 direcciones, incluyendo diagonales)
        directions = [(-1,-1), (-1,0), (-1,1),
                      (0,-1),         (0,1),
                      (1,-1),  (1,0), (1,1)]
        # Aleatorizamos el orden de las direcciones
        np.random.shuffle(directions)

        i = 0
        next_pos = None  # Al inicio no hay movimiento seleccionado
        # Iteramos sobre las direcciones hasta encontrar una posición válida
        while i < len(directions) and next_pos is None:
            d = directions[i]
            cand = pos + np.array(d)   # Calculamos la posición candidata
            if self.enviroment.is_inside(cand):  # Verificamos que esté dentro de la cuadrícula
                next_pos = tuple(cand)           # Guardamos como la siguiente posición
            i += 1

        # Si se encontró una celda válida, movemos al agente hacia esa posición
        if next_pos is not None:
            self.enviroment.move_to(self, next_pos)

    def sense_and_act(self):
        # El agente percibe su celda actual
        pos = tuple(self.getpos())
        # Si la celda está sucia (1), la limpia (cambia a 0)
        if self.enviroment.dirtgrid[pos] == 1:
            self.enviroment.dirtgrid[pos] = 0
            self.cleaned = True
        else:
            # Si la celda ya está limpia, el agente se mueve de forma aleatoria
            self.cleaned = False
            self.moverandom()


# Definición del entorno (habitación)
class Room(ap.Grid):

    def setup(self):
        # Porcentaje de celdas sucias definido en los parámetros
        dirtporcen = self.model.p.dirtporcen
        # Creamos una cuadrícula de ceros (todo limpio al inicio)
        self.dirtgrid = np.zeros(self.shape, dtype=int)
        # Calculamos cuántas celdas estarán sucias
        n_dirty = int(dirtporcen * self.shape[0] * self.shape[1])
        # Lista de todas las posiciones de la cuadrícula
        allpos = [(i,j) for i in range(self.shape[0]) for j in range(self.shape[1])]
        # Seleccionamos al azar las posiciones que estarán sucias
        dirtpos = np.random.choice(len(allpos), n_dirty, replace=False)
        for idx in dirtpos:
            x, y = allpos[idx]
            self.dirtgrid[x, y] = 1  # Marcamos la celda como sucia

    def is_inside(self, pos):
        # Verifica si una posición (x,y) está dentro de los límites de la cuadrícula
        return 0 <= pos[0] < self.shape[0] and 0 <= pos[1] < self.shape[1]


# Definición del modelo de simulación
class VacuumModel(ap.Model):

    def setup(self):
        # Se crea el entorno (Room), usando el tamaño definido en parámetros
        self.enviroment = Room(self, self.p.shape)
        self.enviroment.setup()  # Inicializamos la suciedad
        # Creamos la lista de agentes aspiradora
        self.agents = ap.AgentList(self, self.p.num_agents, VacuumAgent)
        # Colocamos a los agentes dentro del entorno
        self.enviroment.add_agents(self.agents)

    def step(self):
        # En cada paso de la simulación, cada agente percibe y actúa
        self.agents.sense_and_act()


# Función de visualización
def plot_vacuum(model, ax):
    ax.clear()  # Limpiamos la figura anterior
    grid = model.enviroment.dirtgrid
    M, N = grid.shape

    # Mostramos la cuadrícula en escala de grises (1 = sucio, 0 = limpio)
    ax.imshow(grid, cmap='Greys', origin='upper', vmin=0, vmax=1)

    # Dibujamos a los agentes como puntos rojos
    for agent in model.agents:
        pos = model.enviroment.positions[agent]
        ax.plot(pos[1], pos[0], 'ro')

    # Configuramos las marcas de la cuadrícula
    ax.set_xticks(range(N))
    ax.set_yticks(range(M))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    # Título con el número de paso actual
    ax.set_title(f"Actividad {model.t}")
# Parámetros de la simulación

parameters = {
    'steps': 40,        # Número de pasos totales de la simulación
    'shape': (5,5),     # Tamaño de la habitación (5x5)
    'num_agents': 2,    # Número de agentes aspiradora
    'dirtporcen': 0.3   # Porcentaje inicial de celdas sucias
}


# Ejecución de la simulación

fig, ax = plt.subplots(figsize=(5,5))       # Creamos la figura para graficar
model = VacuumModel(parameters)             # Instanciamos el modelo
animation = ap.animate(model, fig, ax, plot_vacuum)  # Animación paso a paso
HTML(animation.to_jshtml())                 # Mostramos la animación en el notebook